# 04 — Component Evaluation

> **Requires a free API key.** Set `GROQ_API_KEY` or `GOOGLE_API_KEY` before running.

## Overview

This notebook runs the full skill-extraction + ESCO-matching pipeline against a
labelled gold set (`data/fixtures/gold_skills.json`) and reports:

- **Macro precision, recall, and F1** across all entries.
- A per-entry table showing predicted skills, gold skills, missed skills, and extra
  (false-positive) skills.

The gold set was hand-annotated: each entry is a short job-description excerpt paired
with the correct set of ESCO preferred labels.

In [ ]:
# fmt: off
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside the project tree.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')
# fmt: on


## 1. Set up the pipeline

In [ ]:
from src.data.esco_loader import EscoIndex
from src.skills.esco_matcher import EscoMatcher
from src.generation.llm_client import build_chat_model, simple_complete
from src.skills.extractor import extract_skill_phrases

index = EscoIndex.load()
matcher = EscoMatcher(index)
chat_model = build_chat_model(temperature=0.0)

def extract_fn(text: str) -> list[str]:
    return extract_skill_phrases(text, lambda p: simple_complete(p, chat_model))

print(f'Index size: {len(index.labels)} skills')

## 2. Run evaluate()

In [ ]:
from src.eval.component_eval import evaluate

gold_path = 'data/fixtures/gold_skills.json'
results = evaluate(gold_path, extract_fn, matcher.match)

print('=== Macro metrics ===')
print(f"Precision : {results['macro_precision']:.4f}")
print(f"Recall    : {results['macro_recall']:.4f}")
print(f"F1        : {results['macro_f1']:.4f}")
print(f"Entries   : {results['n']}")

## 3. Per-entry breakdown

In [ ]:
import json
import pandas as pd
from pathlib import Path
from src.eval.component_eval import prf1

gold = json.loads(Path(gold_path).read_text(encoding='utf-8'))

rows = []
for entry in gold:
    predicted = set(matcher.match(extract_fn(entry['description'])))
    gold_set = set(entry['esco_skills'])
    p, r, f = prf1(predicted, gold_set)
    rows.append({
        'description_snippet': entry['description'][:60] + '...',
        'gold_skills': ', '.join(sorted(gold_set)),
        'predicted_skills': ', '.join(sorted(predicted)),
        'missed': ', '.join(sorted(gold_set - predicted)),
        'extra': ', '.join(sorted(predicted - gold_set)),
        'precision': round(p, 3),
        'recall': round(r, 3),
        'f1': round(f, 3),
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80)
df